# 03. Feature Engineering
41 delta features for matchup prediction.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

fm = pd.read_parquet('../data/processed/feature_matrix.parquet')
print(f'Feature matrix shape: {fm.shape}')
fm.head(3)

## Feature inventory

In [ ]:
delta_cols = [c for c in fm.columns if c.endswith('_delta')]
h2h_cols = [c for c in fm.columns if c.startswith('h2h_')]
extra_cols = ['surface_grass']
feature_cols = delta_cols + h2h_cols + extra_cols

print(f'Delta features:  {len(delta_cols)}')
print(f'H2H features:    {len(h2h_cols)}')
print(f'Extra features:  {len(extra_cols)}')
print(f'Total features:  {len(feature_cols)}')
print()

inventory = pd.DataFrame({
    'column': feature_cols,
    'dtype': [str(fm[c].dtype) for c in feature_cols],
    'non_null': [fm[c].notna().sum() for c in feature_cols],
    'null_pct': [(fm[c].isna().sum() / len(fm) * 100).round(1) for c in feature_cols],
})
print(inventory.to_string(index=False))

## Elo distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col, color, label in zip(
    axes,
    ['elo_delta', 'welo_delta'],
    ['steelblue', 'seagreen'],
    ['Overall Elo delta (winner − loser)', 'Weighted Elo delta (grass-adjusted)']
):
    data = fm[col].dropna()
    ax.hist(data, bins=60, color=color, alpha=0.8, edgecolor='white', linewidth=0.3)
    ax.axvline(0, color='red', linestyle='--', linewidth=1.2, label='zero')
    ax.axvline(data.mean(), color='orange', linestyle='--', linewidth=1.2,
               label=f'mean={data.mean():.1f}')
    ax.set_title(label)
    ax.set_xlabel('Delta (winner - loser)')
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)

plt.suptitle('Elo Rating Deltas (positive = winner had higher rating)', y=1.02)
plt.tight_layout()
plt.show()

for col in ['elo_delta', 'welo_delta']:
    d = fm[col].dropna()
    print(f'{col}: mean={d.mean():.1f}, std={d.std():.1f}, positive={( d > 0).mean():.1%}')

## Correlation matrix

In [ ]:
# Top 20 features by absolute correlation with y
feature_data = fm[feature_cols + ['y']].fillna(0)
corr_with_y = feature_data.corr()['y'].drop('y').abs().nlargest(20)
top20 = list(corr_with_y.index)

corr_matrix = feature_data[top20].corr()

fig, ax = plt.subplots(figsize=(13, 11))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    ax=ax,
    annot_kws={'size': 7}
)
ax.set_title('Correlation Matrix — Top 20 Features (by |corr| with y)', fontsize=12)
plt.tight_layout()
plt.show()

## Key feature distributions

In [ ]:
key_features = ['rank_delta', 'win_streak_delta', 'h2h_win_pct_w']
titles = [
    'rank_delta (winner rank − loser rank)\n(negative = winner ranked higher)',
    'win_streak_delta\n(winner streak − loser streak)',
    'h2h_win_pct_w\n(historical H2H win rate for row-winner)'
]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, col, title in zip(axes, key_features, titles):
    data = fm[col].dropna()
    ax.hist(data, bins=50, color='mediumpurple', alpha=0.8, edgecolor='white', linewidth=0.3)
    ax.axvline(data.mean(), color='red', linestyle='--', linewidth=1.2,
               label=f'mean={data.mean():.2f}')
    ax.set_title(title, fontsize=9)
    ax.legend(fontsize=8)

plt.suptitle('Key Feature Distributions', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

for col in key_features:
    d = fm[col].dropna()
    print(f'{col}: n={len(d):,}, mean={d.mean():.3f}, std={d.std():.3f}, min={d.min():.2f}, max={d.max():.2f}')